# Lahore High-Rise Buildings (Open Buildings 2023)
## Export: 250 m aggregated point GeoJSON to Google Drive


### 0. Initialize Earth Engine


In [1]:
import ee
import geemap
import geopandas as gpd

EE_SCOPES = [
    "https://www.googleapis.com/auth/earthengine",
    "https://www.googleapis.com/auth/drive",
    "https://www.googleapis.com/auth/devstorage.full_control",
    "https://www.googleapis.com/auth/cloud-platform",
]
EE_PROJECT = "1035330553052"
FORCE_REAUTH = True  # Set to False after one successful Drive-authorized login.

if FORCE_REAUTH:
    ee.Authenticate(auth_mode="localhost", scopes=EE_SCOPES, force=True)

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate(auth_mode="localhost", scopes=EE_SCOPES)
    ee.Initialize(project=EE_PROJECT)


/Users/ahmed/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(



Successfully saved authorization token.


### 1. Parameters


In [2]:
BOUNDARY_PATH = "../lahore.geojson"
YEAR = 2023
PRESENCE_THRESH = 0.5
HIGHRISE_HEIGHT_M = 18.0
EXPORT_SCALE_M = 250
OUT_PREFIX = f"highrise_lahore_openbuildings_{YEAR}"

print(f"Using Open Buildings annual snapshot: {YEAR}")
print(f"High-rise threshold: {HIGHRISE_HEIGHT_M} m")
print(f"Output prefix: {OUT_PREFIX}")


Using Open Buildings annual snapshot: 2023
High-rise threshold: 18.0 m
Output prefix: highrise_lahore_openbuildings_2023


### 2. Build Server-Side 250 m Grid and Aggregate Height Metrics


In [3]:
gdf = gpd.read_file(BOUNDARY_PATH)[["geometry"]]
if gdf.crs is None:
    gdf = gdf.set_crs(4326)
if gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(4326)
gdf = gdf.dissolve().reset_index(drop=True)
gdf["geometry"] = gdf["geometry"].buffer(0)

region = geemap.gdf_to_ee(gdf).geometry()
grid_proj = ee.Projection("EPSG:32643").atScale(EXPORT_SCALE_M)
grid_fc = region.coveringGrid(grid_proj, EXPORT_SCALE_M)
grid_fc = grid_fc.map(
    lambda f: ee.Feature(
        f.geometry(),
        {
            "cell_id": f.id(),
            "area_ha": f.geometry().area(1).divide(10000),
        },
    )
)

collection = (
    ee.ImageCollection("GOOGLE/Research/open-buildings-temporal/v1")
    .filterBounds(region)
    .filterDate(f"{YEAR}-01-01", f"{YEAR + 1}-01-01")
)
template = ee.Image(collection.first())
presence_proj = template.select("building_presence").projection()
height_proj = template.select("building_height").projection()
mosaic = collection.mosaic()
presence = mosaic.select("building_presence").setDefaultProjection(presence_proj).clip(region)
height = mosaic.select("building_height").setDefaultProjection(height_proj).clip(region)
built_mask = presence.gte(PRESENCE_THRESH)
highrise = built_mask.And(height.gte(HIGHRISE_HEIGHT_M)).rename("highrise_share")
height_built = height.updateMask(built_mask)

height_stats = height_built.reduceRegions(
    collection=grid_fc,
    reducer=ee.Reducer.mean(),
    scale=4,
    tileScale=4,
)
height_stats = height_stats.map(
    lambda f: ee.Feature(f.geometry(), f.toDictionary())
        .set("height_mean", f.get("mean"))
        .select(["cell_id", "area_ha", "height_mean"], None, False)
)

highrise_stats = highrise.reduceRegions(
    collection=grid_fc,
    reducer=ee.Reducer.mean(),
    scale=4,
    tileScale=4,
)
highrise_stats = highrise_stats.map(
    lambda f: ee.Feature(f.geometry(), f.toDictionary())
        .set("highrise_share", f.get("mean"))
        .select(["cell_id", "highrise_share"], None, False)
)

joined = ee.Join.inner().apply(
    height_stats,
    highrise_stats,
    ee.Filter.equals(leftField="cell_id", rightField="cell_id"),
)

stats_fc = ee.FeatureCollection(
    joined.map(
        lambda pair: ee.Feature(
            ee.Feature(pair.get("primary")).geometry().centroid(1),
            ee.Feature(pair.get("primary")).toDictionary().combine(
                ee.Feature(pair.get("secondary")).toDictionary(),
                overwrite=True,
            ),
        )
    )
)


### 3. Export Aggregated Point GeoJSON to Google Drive


In [4]:
export_prefix = f"{OUT_PREFIX}_points_{EXPORT_SCALE_M}m"
task = ee.batch.Export.table.toDrive(
    collection=stats_fc,
    description=export_prefix,
    folder="EE_Exports",
    fileNamePrefix=export_prefix,
    fileFormat="GeoJSON",
)
task.start()

print(f"Started Drive export: EE_Exports/{export_prefix}.geojson")
print("Check task status in the Earth Engine Tasks tab or with task.status().")


Started Drive export: EE_Exports/highrise_lahore_openbuildings_2023_points_250m.geojson
Check task status in the Earth Engine Tasks tab or with task.status().
